Seojin

# Notice

You can visualize plot only in mac. I tested vedo in Ubuntu, but It did not work.

# Vedo(Visualization of эD Objects)

Vedo is scientific tool for visualizing 3d objects

Official site: https://vedo.embl.es  
Github: https://github.com/marcomusy/vedo/tree/master/examples/notebooks

# Mesh

For visualizing 3d Object, You should make mesh(vertexes) based on nifti image. For making vtk file, I used nii2mesh.

Github: https://github.com/neurolabusc/nii2mesh

ex) nii2mesh lt_hippo_fan.nii lt_hippo_fan_d.vtk

# Library

In [1]:
import nilearn.plotting
import nibabel as nb
import vedo
from matplotlib import cm
import numpy as np
import os
from vedo import *

# Custom function

custom function for converting coordinate system.


In [2]:
def LPSp_toRASp(xyz):
    """
    Convert LPS+ coordinate to RAS+ coordinate
    
    :param xyz: LPS+ coord(list)
    
    return xyz(list)
    """
    x = xyz[0]
    y = xyz[1]
    z = xyz[2]
    
    return -x, -y, z

def RASp_toLPSp(xyz):
    """
    Convert RAS+ coordinate to LPS+ coordinate
    
    :param xyz: RAS+ coord(list)
    
    return xyz(list)
    """
    x = xyz[0]
    y = xyz[1]
    z = xyz[2]
    
    return -x, -y, z

def reference2imageCoord(xyz, affine):
    """
    change reference coordinate to image coordinate
    reference coordinate can be scanner coordinate or MNI coordinate...
    
    :param xyz: anatomical coordinate(np.array)
    :param affine: affine matrix(np.array)
    
    return image coordinate(np.array)
    """
    
    result = np.matmul(np.linalg.inv(affine), [xyz[0], xyz[1], xyz[2], 1])[0:3]
    result = np.ceil(result).astype(int) # note: This is ad-hoc process - (np.ceil)
    return result

# Paths

In [3]:
dir_path = "/home/kjh/Desktop/GP/scripts/figures/GP_vis"
lt_putamen_nii_path = f"{dir_path}/putamen_left.nii"
lt_putamen_vtk_path = f"{dir_path}/putamen_left.vtk"

rt_putamen_nii_path = f"{dir_path}/putamen_right.nii"
rt_putamen_vtk_path = f"{dir_path}/putamen_right.vtk"

os.system(f"nii2mesh {lt_putamen_nii_path} -i b {lt_putamen_vtk_path}")
os.system(f"nii2mesh {rt_putamen_nii_path} -i b {rt_putamen_vtk_path}")

target_vtk_volume_paths = [
    lt_putamen_vtk_path,
    rt_putamen_vtk_path,
]

# Stat map
stat_file_path = f"{dir_path}/r03.DLPFC_cTBS.n17.nii"
stat = nb.load(stat_file_path)
stat_affine = stat.affine
print("stat-map shape:", stat.shape)

Unable to find a file named /home/kjh/Desktop/GP/scripts/GP_vis/putamen_left.nii
Unable to find a file named /home/kjh/Desktop/GP/scripts/GP_vis/putamen_right.nii


FileNotFoundError: No such file or no access: '/home/kjh/Desktop/GP/scripts/GP_vis/r03.DLPFC_cTBS.n17.nii'

# Load Data

In [4]:
# Load Vertex files - (Before skipping next part, you need to identify coordinate about data. 
# In my case, they are RAS+ coordinate
# target_vtk_volumes - volume for visualizing stat
target_vtk_volumes = [vedo.load(vtk_path) for vtk_path in target_vtk_volume_paths]

# Merge target volumes
target_vtk_volume = merge(target_vtk_volumes)

t_stat_index = 1
t_stats = stat.get_fdata()[:,:,:,0, t_stat_index] # Select t-stat

[vedo.file_io] ERROR: in load(), cannot load /home/kjh/Desktop/GP/scripts/GP_vis/putamen_left.vtk
[vedo.file_io] ERROR: in load(), cannot load /home/kjh/Desktop/GP/scripts/GP_vis/putamen_right.vtk


NameError: name 'stat' is not defined

# Visualize configuration


In [14]:
# Color map
color_len = 8

jet = cm.get_cmap('jet')
jet_colors = jet(np.linspace(0, 1, color_len))[:,:-1]

alphas = np.repeat(1, color_len) # np.arange(1, 1, 0.1) # color Alpha

/tmp/ipykernel_136937/27086356.py:4: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  jet = cm.get_cmap('jet')


# Merge target volumes

In [18]:
target_stats = []
for i, vertex in enumerate(target_vtk_volume.points()):
    RASp_coord = vertex
    image_coord = reference2imageCoord(RASp_coord, 
                                       affine = stat_affine).astype(int)

    target_stats.append(t_stats[image_coord[0], image_coord[1], image_coord[2]])
    

In [19]:
# reverse y-axis
target_vtk_volume = target_vtk_volume.clone(deep=False).mirror("y")

# Show

In [22]:
import vedo

# Assuming 'target_vtk_volume', 'jet_colors', 'target_stats', 'alphas' are defined
# Apply the colormap and alpha to the volume
target_vtk_volume.cmap(jet_colors, target_stats, alpha=alphas)

# Show the volume with a scalar bar and other parameters set
# Here, the scalar bar addition is managed through the show function parameters directly
vedo.show(target_vtk_volume, __doc__, viewup="z", axes=1, scalarBar=True).close()


TypeError: show() got an unexpected keyword argument 'scalarBar'

In [20]:
# Apply all configuration on volume
target_vtk_volume.cmap(jet_colors, target_stats, alpha=alphas).addScalarBar()

# Show
vedo.show(target_vtk_volume, __doc__, viewup="z", axes=1).close()

AttributeError: 'Mesh' object has no attribute 'addScalarBar'